# 03 — Gate 2: one-detector probe (e5 + XGBoost), direct -> indirect
The cheap test of the thesis on REAL data. Train the cheapest competent detector on the **direct** set (deepset), freeze the operating threshold on the source, apply it unchanged to **indirect** attacks (BIPIA), and run everything through `metrics.py`.

**Question:** does S jump and the source-threshold FNCR collapse while AUROC roughly holds? Yes -> core effect is real, scale up. No -> rethink before investing in the full panel.

**GPU recommended** (Runtime -> Change runtime type -> T4) for the e5 embeddings.

## Session bootstrap

In [1]:
# === SESSION BOOTSTRAP (run first, every session) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
print('Session ready - identity set, src/ on path, repo pulled.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready - identity set, src/ on path, repo pulled.


## Dependencies + device

In [2]:
!pip install -q sentence-transformers xgboost scikit-learn datasets
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

CUDA: True | Tesla T4


## Load data + embed (cached to data/, gitignored)
Frozen e5-base-v2 embeddings. First run computes + caches them; re-runs reload.

In [ ]:
import importlib, data_loaders, detectors, metrics
for m in (data_loaders, detectors, metrics): importlib.reload(m)
from data_loaders import load_all
from detectors import embed_e5, train_xgb, predict_p, threshold_at_fpr, auroc
from metrics import all_metrics, severity_S, bootstrap_ci
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split

df, _ = load_all(include_bipia=True)
deepset = df[df.source=='deepset'].reset_index(drop=True)
bipia   = df[df.source=='bipia'].reset_index(drop=True)
print('deepset', len(deepset), '| bipia', len(bipia))

os.makedirs('data', exist_ok=True)
def cached_embed(texts, path):
    if os.path.exists(path):
        print('load cached', path); return np.load(path)
    emb = embed_e5(list(texts)); np.save(path, emb); return emb
E_deepset = cached_embed(deepset.text, 'data/emb_deepset.npy')
E_bipia   = cached_embed(bipia.text,   'data/emb_bipia.npy')
print('emb shapes', E_deepset.shape, E_bipia.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

## Train on direct (deepset), freeze threshold at 1% FPR on source
Threshold set on source benigns then frozen — this is the deployment-realistic source-threshold transfer (cite Gate AI 2606.02959). *Probe caveat: threshold set in-sample on train benigns; final version uses out-of-fold predictions.*

In [ ]:
idx = np.arange(len(deepset))
tr, te = train_test_split(idx, test_size=0.3, stratify=deepset.label, random_state=0)
Xtr, ytr = E_deepset[tr], deepset.label.values[tr]
Xte, yte = E_deepset[te], deepset.label.values[te]

clf = train_xgb(Xtr, ytr, seed=0)
p_tr = predict_p(clf, Xtr)
t = threshold_at_fpr(p_tr[ytr==0], target_fpr=0.01)
print(f'frozen source threshold t = {t:.4f}')

## Source vs target at the FROZEN threshold
Target = BIPIA indirect attacks + the same source-test benigns, so only the ATTACK distribution shifts (benign side held constant).

In [ ]:
p_s = predict_p(clf, Xte)
m_s = all_metrics(yte, p_s, t=t); m_s['AUROC'] = auroc(yte, p_s)

ben = yte == 0
X_tgt = np.vstack([Xte[ben], E_bipia])
y_tgt = np.r_[np.zeros(ben.sum()), np.ones(len(E_bipia))]
p_t = predict_p(clf, X_tgt)
m_t = all_metrics(y_tgt, p_t, t=t); m_t['AUROC'] = auroc(y_tgt, p_t)

rows = ['AUROC','FNR','S','FNCR','HCFN','ECE_atk','ECE_pooled']
cmp = pd.DataFrame({'source(direct)':[m_s[k] for k in rows],
                    'target(indirect)':[m_t[k] for k in rows]}, index=rows).round(4)
print(f'frozen t = {t:.4f}  | source n_attack={m_s["n_attacks"]}, target n_attack={m_t["n_attacks"]}\n')
print(cmp.to_string())

## Bootstrap 95% CIs on S (source vs target)

In [ ]:
s_pt, s_lo, s_hi = bootstrap_ci(severity_S, yte, p_s, n_boot=1000, t=t)
t_pt, t_lo, t_hi = bootstrap_ci(severity_S, y_tgt, p_t, n_boot=1000, t=t)
print(f'S source {s_pt:.3f} [{s_lo:.3f}, {s_hi:.3f}]')
print(f'S target {t_pt:.3f} [{t_lo:.3f}, {t_hi:.3f}]')

## Figure + save report

In [ ]:
import matplotlib.pyplot as plt
keys = ['FNR','S','FNCR','ECE_atk']
xs = np.arange(len(keys)); w = 0.35
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(xs-w/2, [m_s[k] for k in keys], w, label='source (direct)')
ax.bar(xs+w/2, [m_t[k] for k in keys], w, label='target (indirect)')
ax.set_xticks(xs); ax.set_xticklabels(keys); ax.set_ylabel('value')
ax.set_title('Gate 2: direct -> indirect at frozen source threshold'); ax.legend()
plt.tight_layout(); os.makedirs('figures', exist_ok=True)
plt.savefig('figures/gate2_source_vs_target.png', dpi=150); plt.show()
os.makedirs('reports', exist_ok=True)
cmp.to_csv('reports/gate2_source_vs_target.csv')
print('saved figures/gate2_source_vs_target.png + reports/gate2_source_vs_target.csv')

## Verdict

In [ ]:
dA = m_s['AUROC'] - m_t['AUROC']
print('INTERPRETATION')
print(f"  AUROC : {m_s['AUROC']:.3f} -> {m_t['AUROC']:.3f}  (drop {dA:.3f})")
print(f"  S     : {m_s['S']} -> {m_t['S']}")
print(f"  FNCR  : {m_s['FNCR']:.3f} -> {m_t['FNCR']:.3f}")
core = (m_t['FNCR'] > m_s['FNCR']) and (np.isnan(m_t['S']) or m_t['S'] > 0.5)
if core:
    print('\n  -> CORE EFFECT PRESENT: at the frozen operating point, indirect')
    print('     attacks slip through confidently (S high, FNCR up).')
    if dA < 0.15:
        print('     AUROC roughly holds -> calibration/operating-point failure,')
        print('     not just discrimination. Strongest form of the thesis. SCALE UP.')
    else:
        print('     AUROC also drops -> mixed discrimination+calibration failure.')
else:
    print('\n  -> Effect weak on this pair. Inspect scores before the full panel.')

---
## Commit + push

In [ ]:
!git add -A
!git commit -m "Gate 2: e5+XGBoost direct->indirect transfer at frozen threshold"
!git push
print('Pushed. Confirm GitHub + Drive in sync.')